In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib as plt
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
import numpy as np

# Import Dataset

In [ ]:
train = pd.read_csv("../data/df01.csv")
test = pd.read_csv("../data/df02.csv")

# Dataset Exploration

## Dataset Summary

In [ ]:
train.shape

In [ ]:
train.head(1)

There are 18 columns in this dataset which are divided into four sections: (1) client information (age to loan), (2) current campaigning details (contact until duration and date), (3) previous campaigning details (campaign until poutcome), and (4) the variable of interest: subscription (y).  For this project, modeling will be done based on client information with the train-test split based on when the contacting during the current campaign was done. The dataset ranges from May 2008 until November 2010, where the year of contacting is not shown in the dataset, however, the dataset is in chronological order.

I will not be focusing on details from previous campaigns since a lot in the train set were first-time contacts as shownin the dataset exploration (most of the pdays are -1 meaning that the person has never been contacted on previous campaigns before). In addition, I will be assuming that the time of contact does not impact the likelihood of subscription.

In [ ]:
train.info()

In [ ]:
train.describe().T

"pdays" and "previous" have a lot of the same values as the 3rd quartile is equal to the minimum, these columns (or the whole previous campaigning group of columns) and may be dropped for feature simplicity. 

## Univariate Distributions

In [ ]:
sns.histplot(train['age'], bins = 10)

In [ ]:
sns.histplot(train['balance'])

In [ ]:
sns.histplot(train[(train['balance'] < 10000) & (train['balance'] > -2000)]['balance'])

In [ ]:
sns.histplot(train['duration'])

In [ ]:
sns.histplot(train['campaign'])

In [ ]:
sns.histplot(train['pdays'])

In [ ]:
sns.histplot(train[train['pdays'] != -1]['pdays'])

In [ ]:
sns.histplot(train['previous'])

In [ ]:
sns.histplot(train[train['previous'] > 0]['previous'])

Most of the (non-temporal) numeric columns are skewed to the right, which may require transformrations to reduce the outlying values for model training

## Bivariate Distirbutions

In [ ]:
sns.pairplot(train)

In [ ]:
sns.heatmap(train.corr(numeric_only = True))

From the pairplot, most of the points are scattered rectangularly, meaning there is no linear relationship, however, from the correlation heatmap, pdays and previous have moderate correlation, which may be a problem with regression models. These columns should be considered for joint transformation or feature extraction to reduce information redundancy and multicollinearity.

In [ ]:
train.describe(exclude='number').T

In [ ]:
sns.countplot(train, y = 'marital', order = train['marital'].value_counts().index)

In [ ]:
sns.countplot(train, y = 'job', order = train['job'].value_counts().index)

There is a significant decrease in counts when it comes to retired, entrepeneur, self-employed, housemaid, unemployed, and student. It may be worth considering to create groupings to avoid count imbalances during model training, and to reduce the number of features after dummy encoding.

In [ ]:
sns.countplot(train, y = 'education', order = train['education'].value_counts().index)

In [ ]:
sns.countplot(train, y = 'contact', order = train['contact'].value_counts().index)

# Feature Engineering

y (marketing outcome) is currently a string, so i convert it into a boolean

In [ ]:
train['y'] = train['y'] == "yes"
test['y'] = test['y'] == "yes"

I choose to group the job descriptions with the following mapping to address groups with relatively low counts

In [ ]:
job_mapping = {
    'management': 'management',
    'technician': 'technician',
    'admin.': 'admin.',
    'services': 'services',
    'blue-collar': 'blue-collar',
    'entrepreneur': 'business-owner',
    'self-employed': 'business-owner',
    'retired': 'non-working',
    'unemployed': 'non-working',
    'student': 'non-working',
    'housemaid': 'services',
    'unknown': 'unknown'
}

train['job_group'] = train['job'].map(job_mapping)
test['job_group'] = test['job'].map(job_mapping)

visualization of new job column

In [ ]:
sns.countplot(train, y = 'job_group', order = train['job_group'].value_counts().index)

choosing relevant variables, remove prevcious campaigning informaiton (high missingness). remove temporal variables

In [ ]:
retained_variables = ['age','job_group','education','default','balance','housing','loan','contact','y']
train = train[retained_variables]
test = test[retained_variables]

scaling. robustscaling was chosen due to skewness of data with high outliers

In [ ]:
numeric_cols = train.select_dtypes(include=['number']).columns
scaler = RobustScaler()
train[numeric_cols] = scaler.fit_transform(train[numeric_cols])
test[numeric_cols] = scaler.transform(test[numeric_cols])

In [ ]:
train = pd.get_dummies(train, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

In [ ]:
train.columns

In [ ]:
X_train = train.drop(columns = 'y')
y_train = train.y
X_test = test.drop(columns = 'y')
y_test = test.y

balancing using SMOTE resampling

In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [ ]:
train.shape

In [ ]:
train.head(1)

# Model Training

# Model Building using LogisticRegression

In [ ]:
X_test['contact_unknown'] = False

In [ ]:
lr = LogisticRegression()
lr.fit(X_train_resampled, y_train_resampled)
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))

# Model Building using XGBoost

In [ ]:
xgb = XGBClassifier()
xgb.fit(X_train_resampled, y_train_resampled)

In [ ]:
y_pred = xgb.predict(X_test)
print(classification_report(y_test, y_pred))

Looking at the (True) recall rates, clasifier performed better than the XGBoost model. Thus, the default logistic regresison model will be used for deployment

# Model Tuning

In [ ]:
param_grid = [
    {
        'solver': ['lbfgs', 'newton-cg', 'sag'],
        'penalty': ['l2'],
        'C': np.logspace(-4, 4, 9)
    },
    {
        'solver': ['liblinear'],
        'penalty': ['l1', 'l2'],
        'C': np.logspace(-4, 4, 9)
    },
    {
        'solver': ['saga'],
        'penalty': ['l1', 'l2', 'elasticnet'],
        'C': np.logspace(-4, 4, 9),
        'l1_ratio': [0.5]  # Only required if penalty is 'elasticnet'
    }
]

grid_search = GridSearchCV(estimator=lr, param_grid=param_grid, cv=5, scoring='recall', n_jobs=-1)
grid_search.fit(X_train_resampled, y_train_resampled)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best CV Recall Score: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

In [ ]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

Thus, the best hyperparameters is solved with a cross-validation recall score of 0.8. However, it showed a 100% recall score, which is suspicious.

I will use the basic logistic regression classifier for deployment since the goal of this project is model deployment more than model development

Now, the contents of these notebooks are modularized.